# **Testing and Exploration**
*Interactive Jupyter notebook for testing and experimenting with Celery application.*

## ***Before you start***

You can always connect directly to the Redis container and manually inspect the database state using standard redis-cli commands like:

---

1. **View all databases:**
   ```sql
   INFO keyspace
   ```

---

2. **View keys in current keyspace:**
   ```sql
   SELECT <database_number> 
   SCAN 0 COUNT 100
   ```
   Or to see all keys at once:
   ```sql
   SELECT <database_number> 
   KEYS *
   ```

---

3. **View all users/roles:**
   ```sql
   ACL LIST
   ```

---

4. **Exit redis-cli:**
   ```sql
   exit
   ```

---

## **Setup Environment**

In [ ]:
import sys
sys.path.append('.')

from celery.result import AsyncResult
from app.task_queue.config import celery_config
from app.services import MockTaskService

service = MockTaskService()

In [ ]:
def check_tasks_status(tasks: list[AsyncResult]):
    """
    Check the status and results of the provided tasks.

    This function iterates through the list of AsyncResult objects
    and prints the current status of each task. If a task is completed,
    it prints the result or error information accordingly.

    Parameters
    ----------
    tasks : list[celery.result.AsyncResult]
        A list of AsyncResult objects representing the tasks to check.
        These objects should be obtained from task dispatching functions
        such as run_cpu_tasks or similar task submission mechanisms.

    Notes
    -----
    For each task, this function prints:
    - Task ID
    - Current status (PENDING, STARTED, SUCCESS, FAILURE, etc.)
    - Result if the task is completed successfully
    - Error information if the task failed
    """

    print("\n--- Task Status ---")
    for i, task in enumerate(tasks):
        print(f"Task {i+1} (ID: {task.id}):")
        print(f"  Status: {task.status}")
        if task.ready():
            if task.successful():
                print(f"  Result: {task.result}")
            else:
                print(f"  Error: {task.result}")
        else:
            print("  Task is still running...")
        print()

In [ ]:
# Start Redis
!pixi run just redis-up

In [ ]:
# # After finishing your work, shut down Redis to free resources
# !pixi run just redis-down

## **Test**

> **Note:**  
It's normal to see yellow warnings and red error messages in the terminal/worker logs. These tasks are designed to randomly fail to test retry logic — this is expected behavior.

Start celery workers:

```bash
pixi run worker-cpu-up
pixi run worker-io-up
```

> **Important:**  
Do NOT run these commands directly in this Jupyter notebook  as magic commands because they will block the kernel (the worker runs indefinitely), preventing you from executing any further cells. Instead, run the workers in a separate terminals, then use this notebook to send tasks and monitor their execution.

### **I. CPU-bound task**

In [ ]:
def run_cpu_tasks():
    """
    Launch 3 CPU-intensive tasks and return their AsyncResult objects.

    This function dispatches 3 CPU-intensive tasks with varying durations. 
    Each task is submitted to the Celery broker for multiprocessing 
    (default if no changes made to worker configuration) execution by a worker.
    The function returns immediately after submitting the tasks,
    providing AsyncResult objects for tracking their status and results.

    Returns
    -------
    list[celery.result.AsyncResult]
        A list of AsyncResult objects representing the dispatched tasks.
        These objects can be used to check the status and retrieve results
        of the running tasks.
    """

    tasks = []
    for i in range(3):
        # Submit task with varying duration for diversity
        task = service.run_cpu_intensive_task.delay(duration=2 + (i+1) * 2)
        tasks.append(task)
        print(f"Launched task {i+1} with ID: {task.id}")

    return tasks

cpu_tasks = run_cpu_tasks()

In [ ]:
check_tasks_status(cpu_tasks)

### **II. I/O-bound task**

In [ ]:
def run_io_tasks():
    """
    Launch 20 I/O-intensive tasks and return their AsyncResult objects.

    This function dispatches 20 I/O-intensive tasks with varying durations. 
    Each task is submitted to the Celery broker for cooperative multitasking 
    (default if no changes made to worker configuration) execution by a worker.
    The function returns immediately after submitting the tasks,
    providing AsyncResult objects for tracking their status and results.

    Returns
    -------
    list[celery.result.AsyncResult]
        A list of AsyncResult objects representing the dispatched tasks.
        These objects can be used to check the status and retrieve results
        of the running tasks.
    """

    tasks = []
    for i in range(20):
        # Submit task with varying duration for diversity
        task = service.run_io_intensive_task.delay(duration=1 + (i+1) * 0.5)
        tasks.append(task)
        print(f"Launched task {i+1} with ID: {task.id}")

    return tasks

io_tasks = run_io_tasks()

In [ ]:
check_tasks_status(io_tasks)

## **Celery Task Execution Patterns**

| Method        | Description                                                                       | Blocking | Use Case                                      |
|---------------|-----------------------------------------------------------------------------------|----------|-----------------------------------------------|
| `delay`       | Shortcut for `apply_async()` with no additional options.                          | No       | Simple async task dispatch.                   |
| `apply_async` | Asynchronous task dispatch with full control over options (queue, routing, etc.). | No       | Advanced async task dispatch with options.    |
| `apply`       | Synchronous task execution in the current process.                                | Yes      | Testing or when you need immediate result.    |
| `get`         | Retrieve result from an AsyncResult object (blocks until ready by default).       | Yes      | Fetching results after task dispatch.         |

Since the code above used simple `delay` for basic task dispatch, we will now test other methods in scenarios where they are more appropriate, starting with `apply_async` for launching tasks with different priorities.

### **I. `apply_async()`**

In [ ]:
def apply_cpu_tasks():
    """
    Launch 5 CPU-intensive tasks with varying priorities and return their AsyncResult objects.

    This function dispatches 5 CPU-intensive tasks with varying durations and random priorities. 
    Each task is submitted to the Celery broker for multiprocessing 
    (default if no changes made to worker configuration) execution by a worker.
    The function returns immediately after submitting the tasks,
    providing AsyncResult objects for tracking their status and results.

    Returns
    -------
    list[celery.result.AsyncResult]
        A list of AsyncResult objects representing the dispatched tasks.
        These objects can be used to check the status and retrieve results
        of the running tasks.
    """

    tasks = []
    for i in range(5):
        # Assign priority in descending order: max -> max-1 -> ... -> 1
        priority = max(1, celery_config.cpu_bound_queue_max_priority - i)
        
        # Submit task with varying duration and random priority
        task = service.run_cpu_intensive_task.apply_async(
            args=[2 + (i+1) * 2],  # duration argument
            priority=priority
        )
        tasks.append(task)
        print(f"Launched task {i+1} with ID: {task.id} and priority: {priority}")

    return tasks

applied_cpu_tasks = apply_cpu_tasks()

In [ ]:
check_tasks_status(applied_cpu_tasks)

Consider a situation where we have a machine learning model that takes input from the results of five other models; it cannot start until all five responses are received. In this case, we want the main model to also run through Celery workers, so we need to wait until all five responses are obtained before launching it. In such scenarios, the `get()` method is helpful as it blocks until all individual tasks are completed, allowing us to collect all required results before proceeding with the main model task.

### **II. `get()`**

In [ ]:
def get_tasks_status(tasks: list[AsyncResult]):
    """
    Check the status and results of the provided tasks by waiting for completion.

    This function iterates through the list of AsyncResult objects
    and waits for each task to complete using the get() method.
    The function blocks until all tasks are finished and then
    prints the results or error information for each task.

    Parameters
    ----------
    tasks : list[celery.result.AsyncResult]
        A list of AsyncResult objects representing the tasks to check.
        These objects should be obtained from task dispatching functions
        such as run_cpu_tasks or similar task submission mechanisms.

    Notes
    -----
    For each task, this function prints:
    - Task ID
    - Result if the task completed successfully
    - Error information if the task failed
    The function will block until all tasks have completed.
    """

    print("\n--- Task Results ---")
    for i, task in enumerate(tasks):
        print(f"Waiting for task {i+1} (ID: {task.id})...")
        try:
            result = task.get()  # This will block until the task is ready
            print(f"  Result: {result}")
        except Exception as e:
            print(f"  Error: {e}")
        print()

cpu_tasks_for_get = apply_cpu_tasks()
get_tasks_status(cpu_tasks_for_get)

Consider this scenario: you’re writing a unit test for your Celery task logic and want to verify its behaviour in the same process, without involving Redis, workers, or async overhead.  
Using `apply()` lets you call the task like a regular function — no broker, no worker, no delays. Perfect for isolated testing.

### **III. `apply()`**

Even if all workers and Redis are turned off, the code below will still work — because `apply()` executes tasks synchronously within the current process, bypassing the entire Celery broker infrastructure.

In [ ]:
def apply_run_io_tasks():
    """
    Launch 5 I/O-intensive tasks sequentially using apply and return their results.

    This function dispatches 5 I/O-intensive tasks with varying durations using 
    the apply method, which executes tasks synchronously in the current process.
    Although the default worker uses gevent for cooperative multitasking that could 
    execute these tasks in parallel, the apply method forces sequential execution.
    Each task runs one after another, blocking until completion, making it clearly 
    visible that all tasks execute strictly sequentially despite the gevent worker 
    configuration.

    Returns
    -------
    list[dict]
        A list of results from the completed tasks, representing the output
        of each sequentially executed I/O-intensive operation.
    """

    results = []
    for i in range(5):
        # Using apply() forces synchronous execution in current process
        result = service.run_io_intensive_task.apply(args=[7])
        results.append(result)
        print(f"Completed task {i+1} with result: {result}")

    return results

io_results = apply_run_io_tasks()